# Dashboard de Operaciones - Online Retail

## Objetivo

El objetivo de este proyecto es analizar las operaciones de una tienda online y construir indicadores clave de desempeño (KPI) que permitan monitorear ventas, pedidos, clientes y cancelaciones.

El proyecto utiliza Python para limpiar y analizar más de 500 mil registros, y prepara una base que puede utilizarse directamente en Power BI.

### Etapas

- Exploración inicial de los datos.
- Limpieza y validación.
- Definición de ventas válidas y cancelaciones.
- Construcción de KPI operacionales.
- Análisis mensual.
- Análisis por país y producto.
- Preparación de datos para Power BI.
- Exportación de resultados.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RUTA_DATOS = Path("datos/Online Retail.xlsx")
RUTA_RESULTADOS = Path("resultados")
RUTA_RESULTADOS.mkdir(exist_ok=True)

datos = pd.read_excel(RUTA_DATOS)

print("Cantidad de filas:", datos.shape[0])
print("Cantidad de columnas:", datos.shape[1])

datos.head()

## 1. Exploración inicial

Antes de realizar cualquier análisis se revisa la estructura general de la base, los valores faltantes y los registros duplicados.

In [ ]:
datos.info()

print("\nValores faltantes por columna:")
print(datos.isnull().sum())

print("\nFilas duplicadas:")
print(datos.duplicated().sum())

### Primeras observaciones

La base contiene más de 500 mil líneas de transacciones.

Durante la exploración se observan:

- Filas duplicadas.
- Valores faltantes, principalmente en `CustomerID`.
- Cantidades negativas asociadas a devoluciones o ajustes.
- Facturas que comienzan con `C`, utilizadas para identificar cancelaciones.
- Precios iguales o menores que cero que no se consideran ventas válidas.

## 2. Preparación de los datos

Se eliminan únicamente los duplicados exactos y se mantienen las cancelaciones dentro de la base general.

Una venta válida debe cumplir simultáneamente:

- La factura no comienza con `C`.
- La cantidad es mayor que cero.
- El precio unitario es mayor que cero.

In [ ]:
filas_antes = len(datos)

datos = datos.drop_duplicates().copy()
datos["InvoiceNo"] = datos["InvoiceNo"].astype(str)

datos["EsCancelacion"] = datos["InvoiceNo"].str.startswith("C")
datos["Importe"] = datos["Quantity"] * datos["UnitPrice"]

datos["Fecha"] = datos["InvoiceDate"].dt.strftime("%Y-%m-%d")
datos["Año"] = datos["InvoiceDate"].dt.year
datos["NumeroMes"] = datos["InvoiceDate"].dt.month
datos["Mes"] = datos["InvoiceDate"].dt.month_name()
datos["Periodo"] = datos["InvoiceDate"].dt.to_period("M").astype(str)

datos["EsVentaValida"] = (
    (~datos["EsCancelacion"])
    & (datos["Quantity"] > 0)
    & (datos["UnitPrice"] > 0)
)

ventas = datos[datos["EsVentaValida"]].copy()

print("Filas originales:", filas_antes)
print("Filas después de eliminar duplicados:", len(datos))
print("Ventas válidas:", len(ventas))

## 3. KPI operacionales

Los principales indicadores utilizados en el dashboard son:

- Facturación total.
- Cantidad de pedidos.
- Unidades vendidas.
- Ticket promedio.
- Clientes activos.
- Tasa de cancelación.

In [ ]:
facturacion_total = ventas["Importe"].sum()
pedidos = ventas["InvoiceNo"].nunique()
unidades_vendidas = ventas["Quantity"].sum()

importe_por_pedido = ventas.groupby("InvoiceNo")["Importe"].sum()
ticket_promedio = importe_por_pedido.mean()

clientes_activos = ventas["CustomerID"].nunique()

facturas_totales = datos["InvoiceNo"].nunique()
facturas_canceladas = datos.loc[
    datos["EsCancelacion"], "InvoiceNo"
].nunique()

tasa_cancelacion = facturas_canceladas / facturas_totales * 100

kpis = pd.DataFrame({
    "Indicador": [
        "Facturación total",
        "Pedidos",
        "Unidades vendidas",
        "Ticket promedio",
        "Clientes activos",
        "Tasa de cancelación"
    ],
    "Valor": [
        round(facturacion_total, 2),
        pedidos,
        unidades_vendidas,
        round(ticket_promedio, 2),
        clientes_activos,
        round(tasa_cancelacion, 2)
    ]
})

kpis

## 4. Evolución mensual

Se analiza la evolución de las operaciones para identificar tendencias y cambios en el volumen de actividad.

In [ ]:
resumen_mensual = (
    ventas.groupby("Periodo")
    .agg(
        Facturacion=("Importe", "sum"),
        Pedidos=("InvoiceNo", "nunique"),
        Unidades=("Quantity", "sum"),
        Clientes=("CustomerID", "nunique")
    )
    .reset_index()
)

resumen_mensual["TicketPromedio"] = (
    resumen_mensual["Facturacion"] /
    resumen_mensual["Pedidos"]
)

resumen_mensual.round(2)

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(
    resumen_mensual["Periodo"],
    resumen_mensual["Facturacion"],
    marker="o"
)
plt.title("Facturación mensual")
plt.xlabel("Mes")
plt.ylabel("Facturación (£)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(
    resumen_mensual["Periodo"],
    resumen_mensual["Pedidos"],
    marker="o"
)
plt.title("Cantidad de pedidos por mes")
plt.xlabel("Mes")
plt.ylabel("Pedidos")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(
    resumen_mensual["Periodo"],
    resumen_mensual["TicketPromedio"],
    marker="o"
)
plt.title("Ticket promedio mensual")
plt.xlabel("Mes")
plt.ylabel("Ticket promedio (£)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Interpretación temporal

Noviembre de 2011 presenta la mayor facturación y la mayor cantidad de pedidos del período.

Entre septiembre y noviembre se observa un aumento importante en la actividad. Durante estos meses el ticket promedio no aumenta en la misma proporción, por lo que el crecimiento de la facturación se relaciona principalmente con un mayor volumen de pedidos.

Diciembre de 2011 no debe compararse directamente con los meses anteriores, ya que la base contiene información solamente hasta el 9 de diciembre.

## 5. Facturación por país

In [ ]:
top_paises = (
    ventas.groupby("Country")["Importe"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_paises

In [ ]:
plt.figure(figsize=(10, 6))
top_paises.sort_values().plot(kind="barh")
plt.title("Top 10 países por facturación")
plt.xlabel("Facturación (£)")
plt.ylabel("País")
plt.tight_layout()
plt.show()

La facturación se encuentra fuertemente concentrada en Reino Unido, que representa la mayor parte del total de ventas válidas.

## 6. Productos con mayor facturación

Para este ranking se excluyen conceptos administrativos o logísticos como `DOTCOM POSTAGE`, `POSTAGE`, `Manual`, `CARRIAGE`, `BANK CHARGES` y `AMAZON FEE`.

In [ ]:
conceptos_no_producto = {
    "DOTCOM POSTAGE",
    "POSTAGE",
    "Manual",
    "CARRIAGE",
    "BANK CHARGES",
    "AMAZON FEE"
}

ventas_productos = ventas[
    ~ventas["Description"].fillna("").isin(conceptos_no_producto)
]

top_productos = (
    ventas_productos.groupby("Description")["Importe"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_productos

In [ ]:
plt.figure(figsize=(11, 6))
top_productos.sort_values().plot(kind="barh")
plt.title("Top 10 productos por facturación")
plt.xlabel("Facturación (£)")
plt.ylabel("Producto")
plt.tight_layout()
plt.show()

## 7. Exportación para Power BI

Se exporta una base preparada con variables de fecha, importe y banderas para identificar ventas válidas y cancelaciones.

In [ ]:
datos_powerbi = datos[
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "Fecha",
        "Año",
        "NumeroMes",
        "Mes",
        "Periodo",
        "UnitPrice",
        "CustomerID",
        "Country",
        "Importe",
        "EsCancelacion",
        "EsVentaValida"
    ]
].copy()

datos_powerbi = datos_powerbi.rename(columns={
    "InvoiceNo": "Factura",
    "StockCode": "CodigoProducto",
    "Description": "Descripcion",
    "Quantity": "Cantidad",
    "InvoiceDate": "FechaHora",
    "UnitPrice": "PrecioUnitario",
    "CustomerID": "Cliente",
    "Country": "Pais"
})

datos_powerbi.to_csv(
    "datos/retail_operaciones_powerbi.csv",
    index=False,
    encoding="utf-8-sig"
)

datos_powerbi.head()

## 8. Guardar resultados

Las tablas y figuras generadas se guardan en la carpeta `resultados`.

In [ ]:
kpis.to_csv(
    RUTA_RESULTADOS / "kpis_resumen.csv",
    index=False,
    encoding="utf-8-sig"
)

resumen_mensual.to_csv(
    RUTA_RESULTADOS / "resumen_mensual.csv",
    index=False,
    encoding="utf-8-sig"
)

plt.figure(figsize=(11, 5))
plt.plot(resumen_mensual["Periodo"], resumen_mensual["Facturacion"], marker="o")
plt.title("Facturación mensual")
plt.xlabel("Mes")
plt.ylabel("Facturación (£)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(RUTA_RESULTADOS / "01_facturacion_mensual.png", dpi=160, bbox_inches="tight")
plt.close()

plt.figure(figsize=(11, 5))
plt.plot(resumen_mensual["Periodo"], resumen_mensual["Pedidos"], marker="o")
plt.title("Cantidad de pedidos por mes")
plt.xlabel("Mes")
plt.ylabel("Pedidos")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(RUTA_RESULTADOS / "02_pedidos_mensuales.png", dpi=160, bbox_inches="tight")
plt.close()

plt.figure(figsize=(11, 5))
plt.plot(resumen_mensual["Periodo"], resumen_mensual["TicketPromedio"], marker="o")
plt.title("Ticket promedio mensual")
plt.xlabel("Mes")
plt.ylabel("Ticket promedio (£)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(RUTA_RESULTADOS / "03_ticket_promedio_mensual.png", dpi=160, bbox_inches="tight")
plt.close()

plt.figure(figsize=(10, 6))
top_paises.sort_values().plot(kind="barh")
plt.title("Top 10 países por facturación")
plt.xlabel("Facturación (£)")
plt.ylabel("País")
plt.tight_layout()
plt.savefig(RUTA_RESULTADOS / "04_top10_paises_facturacion.png", dpi=160, bbox_inches="tight")
plt.close()

plt.figure(figsize=(11, 6))
top_productos.sort_values().plot(kind="barh")
plt.title("Top 10 productos por facturación")
plt.xlabel("Facturación (£)")
plt.ylabel("Producto")
plt.tight_layout()
plt.savefig(RUTA_RESULTADOS / "05_top10_productos_facturacion.png", dpi=160, bbox_inches="tight")
plt.close()

print("Resultados guardados en la carpeta resultados.")